# Laboratory 6: File Handling - Appendix Additional Exercises
**Student:** Solis (`TUPM-26-2631`)
**Course:** Computer Programming 2 (TUP ECE)

This notebook contains the complete implementations and verified outputs for:
1. Web Server Access Log Analysis
2. Mobile Device Usage Profiling
3. Cybersecurity Incident Report


--- 
## 1. Web Server Access Log Analysis


In [1]:
# Student Information & Parameter Derivation
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

RECORDS_LIMIT = 1000 + ID_SUM  # 1020
TARGET_CLASS_DIGIT = (SEED_DIGIT % 4) + 2  # 3 -> 3xx
TARGET_STATUS_CLASS = f"{TARGET_CLASS_DIGIT}xx"

print(f"Student: {LAST_NAME} ({STUDENT_ID})")
print(f"Seed Digit: {SEED_DIGIT}")
print(f"ID Sum: {ID_SUM}")
print(f"Records to Process: {RECORDS_LIMIT}")
print(f"Target Status Class: {TARGET_STATUS_CLASS}")


Student: Solis (TUPM-26-2631)
Seed Digit: 1
ID Sum: 20
Records to Process: 1020
Target Status Class: 3xx


In [2]:
import os
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

# Setup file paths
BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "weblog.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = Path("datasets/weblog.csv")

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_CSV_PATH = OUTPUT_DIR / "weblog_3xx_filtered.csv"
JSON_SUMMARY_PATH = OUTPUT_DIR / "weblog_3xx_summary.json"
BACKUP_JSON_PATH = BACKUP_DIR / "weblog_3xx_summary_backup.json"

print("Dataset Path:", DATASET_PATH.resolve())
print("Output Directory:", OUTPUT_DIR.resolve())
print("Backup Directory:", BACKUP_DIR.resolve())


Dataset Path: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\datasets\weblog.csv
Output Directory: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\output
Backup Directory: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\backup


In [3]:
# Process CSV line-by-line without loading entire dataset into memory
records_processed = 0
records_in_target = 0
invalid_records = 0
status_counts = Counter()
url_counts = Counter()
selected_records = []

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if records_processed >= RECORDS_LIMIT:
            break
        records_processed += 1

        ip = (row.get("IP") or "").strip()
        time = (row.get("Time") or "").strip()
        url = (row.get("URL") or "").strip()
        status = (row.get("Staus") or row.get("Status") or "").strip()

        if not status or not status.isdigit():
            invalid_records += 1
            continue

        if status.startswith(str(TARGET_CLASS_DIGIT)) and len(status) == 3:
            records_in_target += 1
            status_counts[status] += 1
            url_counts[url] += 1
            selected_records.append({
                "IP": ip,
                "Time": time,
                "URL": url,
                "Status": status
            })

top_5_urls = url_counts.most_common(5)

# Export selected records to filtered CSV
fieldnames = ["IP", "Time", "URL", "Status"]
with open(FILTERED_CSV_PATH, mode="w", encoding="utf-8", newline="") as csv_out:
    writer = csv.DictWriter(csv_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected_records)

# Export summary to JSON
summary_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "records_processed": records_processed,
        "target_status_class": TARGET_STATUS_CLASS,
        "records_in_target_class": records_in_target,
        "invalid_or_skipped_records": invalid_records,
        "count_per_status_code": dict(status_counts),
        "five_most_requested_urls": [
            {"rank": i + 1, "url": u, "count": c}
            for i, (u, c) in enumerate(top_5_urls)
        ],
        "filtered_csv_file": str(FILTERED_CSV_PATH.name),
        "json_summary_file": str(JSON_SUMMARY_PATH.name)
    }
}

with open(JSON_SUMMARY_PATH, mode="w", encoding="utf-8") as json_out:
    json.dump(summary_data, json_out, indent=4)

# Create backup and verify
shutil.copy2(JSON_SUMMARY_PATH, BACKUP_JSON_PATH)
backup_verified = BACKUP_JSON_PATH.exists() and BACKUP_JSON_PATH.stat().st_size > 0
backup_size = BACKUP_JSON_PATH.stat().st_size if backup_verified else 0

# Display Assessment Data Table
print("=" * 70)
print(f"{'Item':<35} | {'Result'}")
print("-" * 70)
print(f"{'Records processed':<35} | {records_processed}")
print(f"{'Target status class':<35} | {TARGET_STATUS_CLASS}")
print(f"{'Records in target class':<35} | {records_in_target}")
status_str = ", ".join(f"{k}: {v}" for k, v in sorted(status_counts.items()))
print(f"{'Count per status code':<35} | {status_str}")
print(f"{'Five most requested URLs':<35} |")
for i, (u, c) in enumerate(top_5_urls, 1):
    print(f"  {i}. {u:<50} ({c} requests)")
print(f"{'Filtered CSV file':<35} | {FILTERED_CSV_PATH.name} ({len(selected_records)} rows)")
print(f"{'JSON summary file':<35} | {JSON_SUMMARY_PATH.name}")
print(f"{'Backup verification':<35} | {'VERIFIED' if backup_verified else 'FAILED'} ({BACKUP_JSON_PATH.name}, {backup_size} bytes)")
print("=" * 70)
print("Final Output: Exercise 1 completed successfully.")


Item                                | Result
----------------------------------------------------------------------
Records processed                   | 1020
Target status class                 | 3xx
Records in target class             | 164
Count per status code               | 302: 143, 304: 21
Five most requested URLs            |
  1. GET / HTTP/1.1                                     (45 requests)
  2. POST /process.php HTTP/1.1                         (35 requests)
  3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1 (34 requests)
  4. GET /home.php HTTP/1.1                             (9 requests)
  5. GET /logout.php HTTP/1.1                           (7 requests)
Filtered CSV file                   | weblog_3xx_filtered.csv (164 rows)
JSON summary file                   | weblog_3xx_summary.json
Backup verification                 | VERIFIED (weblog_3xx_summary_backup.json, 1316 bytes)
Final Output: Exercise 1 completed successfully.


--- 
## 2. Mobile Device Usage Profiling


In [4]:
# Student Information & Parameter Derivation
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

TARGET_OS = "Android" if (SEED_DIGIT % 2 == 0) else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1  # 2

EXPORTED_FIELDS = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
]

print(f"Student: {LAST_NAME} ({STUDENT_ID})")
print(f"Assigned OS: {TARGET_OS} (SEED_DIGIT is odd)")
print(f"Assigned Behavior Class: Class {TARGET_BEHAVIOR_CLASS} ((SEED_DIGIT % 5) + 1)")
print(f"Exported Fields: {EXPORTED_FIELDS}")


Student: Solis (TUPM-26-2631)
Assigned OS: iOS (SEED_DIGIT is odd)
Assigned Behavior Class: Class 2 ((SEED_DIGIT % 5) + 1)
Exported Fields: ['User ID', 'Device Model', 'Operating System', 'Battery Drain (mAh/day)', 'Data Usage (MB/day)']


In [5]:
import os
import csv
import json
import shutil
from pathlib import Path

BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "user_behavior_dataset.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = Path("datasets/user_behavior_dataset.csv")

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

EXPORTED_CSV_PATH = OUTPUT_DIR / "user_behavior_ios_class2.csv"
JSON_SUMMARY_PATH = OUTPUT_DIR / "user_behavior_ios_class2_summary.json"
BACKUP_CSV_PATH = BACKUP_DIR / "user_behavior_ios_class2_backup.csv"

print("Dataset Path:", DATASET_PATH.resolve())
print("Exported CSV:", EXPORTED_CSV_PATH.resolve())
print("JSON Summary:", JSON_SUMMARY_PATH.resolve())
print("Backup CSV:  ", BACKUP_CSV_PATH.resolve())


Dataset Path: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\datasets\user_behavior_dataset.csv
Exported CSV: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\output\user_behavior_ios_class2.csv
JSON Summary: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\output\user_behavior_ios_class2_summary.json
Backup CSV:   C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\backup\user_behavior_ios_class2_backup.csv


In [6]:
total_records = 0
records_matching_os = 0
records_in_target_class = 0
selected_users = []
battery_drains = []
data_usages = []

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_records += 1
        os_val = (row.get("Operating System") or "").strip()

        if os_val.lower() == TARGET_OS.lower():
            records_matching_os += 1
            behavior_class_val = (row.get("User Behavior Class") or "").strip()

            if behavior_class_val == str(TARGET_BEHAVIOR_CLASS):
                records_in_target_class += 1

                try:
                    bd = float(row.get("Battery Drain (mAh/day)", 0))
                    battery_drains.append(bd)
                except ValueError:
                    pass

                try:
                    du = float(row.get("Data Usage (MB/day)", 0))
                    data_usages.append(du)
                except ValueError:
                    pass

                user_entry = {field: row.get(field, "").strip() for field in EXPORTED_FIELDS}
                selected_users.append(user_entry)

avg_battery = sum(battery_drains) / len(battery_drains) if battery_drains else 0.0
avg_data = sum(data_usages) / len(data_usages) if data_usages else 0.0

# Export selected users to CSV
with open(EXPORTED_CSV_PATH, mode="w", encoding="utf-8", newline="") as csv_out:
    writer = csv.DictWriter(csv_out, fieldnames=EXPORTED_FIELDS)
    writer.writeheader()
    writer.writerows(selected_users)

# Verify count from written CSV
with open(EXPORTED_CSV_PATH, mode="r", encoding="utf-8") as csv_in:
    exported_reader = csv.DictReader(csv_in)
    exported_row_count = sum(1 for _ in exported_reader)

count_verified = (exported_row_count == records_in_target_class)

# Write summary JSON
summary_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "operating_system_and_behavior_class": f"{TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}",
        "total_records_in_dataset": total_records,
        "records_matching_operating_system": records_matching_os,
        "records_in_target_behavior_class": records_in_target_class,
        "average_battery_drain_mah_day": round(avg_battery, 2),
        "average_data_usage_mb_day": round(avg_data, 2),
        "count_verification": {
            "selected_record_count": records_in_target_class,
            "exported_row_count": exported_row_count,
            "verified": count_verified
        },
        "exported_csv_file": str(EXPORTED_CSV_PATH.name),
        "json_summary_file": str(JSON_SUMMARY_PATH.name)
    }
}

# Create backup of exported CSV
shutil.copy2(EXPORTED_CSV_PATH, BACKUP_CSV_PATH)
backup_exists = BACKUP_CSV_PATH.exists()
backup_file_size = BACKUP_CSV_PATH.stat().st_size if backup_exists else 0

summary_data["assessment_data"]["backup_file_and_size"] = {
    "file_name": str(BACKUP_CSV_PATH.name),
    "size_bytes": backup_file_size,
    "verified": backup_exists
}

with open(JSON_SUMMARY_PATH, mode="w", encoding="utf-8") as json_out:
    json.dump(summary_data, json_out, indent=4)

# Display Assessment Data Table
print("=" * 70)
print(f"{'Item':<40} | {'Result'}")
print("-" * 70)
print(f"{'Operating system and behavior class':<40} | {TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}")
print(f"{'Total records in dataset':<40} | {total_records}")
print(f"{'Records matching operating system':<40} | {records_matching_os}")
print(f"{'Records in target behavior class':<40} | {records_in_target_class}")
print(f"{'Average battery drain (mAh/day)':<40} | {avg_battery:.2f} mAh/day")
print(f"{'Average data usage (MB/day)':<40} | {avg_data:.2f} MB/day")
print(f"{'Count verification':<40} | VERIFIED ({exported_row_count} == {records_in_target_class})")
print(f"{'Exported CSV file':<40} | {EXPORTED_CSV_PATH.name}")
print(f"{'JSON summary file':<40} | {JSON_SUMMARY_PATH.name}")
print(f"{'Backup file and size':<40} | {BACKUP_CSV_PATH.name} ({backup_file_size} bytes)")
print("=" * 70)
print("Final Output: Exercise 2 completed successfully.")


Item                                     | Result
----------------------------------------------------------------------
Operating system and behavior class      | iOS, Class 2
Total records in dataset                 | 700
Records matching operating system        | 146
Records in target behavior class         | 32
Average battery drain (mAh/day)          | 872.03 mAh/day
Average data usage (MB/day)              | 451.34 MB/day
Count verification                       | VERIFIED (32 == 32)
Exported CSV file                        | user_behavior_ios_class2.csv
JSON summary file                        | user_behavior_ios_class2_summary.json
Backup file and size                     | user_behavior_ios_class2_backup.csv (954 bytes)
Final Output: Exercise 2 completed successfully.


--- 
## 3. Cybersecurity Incident Report


In [7]:
# Student Information & Parameter Derivation
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

TARGET_YEAR = 2015 + SEED_DIGIT  # 2016
LOSS_FIELD = "Financial Loss (in Million $)"
COUNT_FIELD = "Attack Type"

print(f"Student: {LAST_NAME} ({STUDENT_ID})")
print(f"Target Year: {TARGET_YEAR} (2015 + SEED_DIGIT)")
print(f"Loss Field: {LOSS_FIELD}")
print(f"Count Field: {COUNT_FIELD}")


Student: Solis (TUPM-26-2631)
Target Year: 2016 (2015 + SEED_DIGIT)
Loss Field: Financial Loss (in Million $)
Count Field: Attack Type


In [8]:
import os
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "Global_Cybersecurity_Threats_2015-2024.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = Path("datasets/Global_Cybersecurity_Threats_2015-2024.csv")

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_CSV_PATH = OUTPUT_DIR / "cybersecurity_2016_filtered.csv"
JSON_TOP5_PATH = OUTPUT_DIR / "cybersecurity_2016_top5_costliest.json"
BACKUP_JSON_PATH = BACKUP_DIR / "cybersecurity_2016_top5_costliest_backup.json"
REPORT_TXT_PATH = OUTPUT_DIR / "cybersecurity_2016_incident_report.txt"

print("Dataset Path:", DATASET_PATH.resolve())
print("Filtered CSV:", FILTERED_CSV_PATH.resolve())
print("JSON Top 5:  ", JSON_TOP5_PATH.resolve())
print("Backup JSON: ", BACKUP_JSON_PATH.resolve())
print("Report TXT:  ", REPORT_TXT_PATH.resolve())


Dataset Path: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\datasets\Global_Cybersecurity_Threats_2015-2024.csv
Filtered CSV: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\output\cybersecurity_2016_filtered.csv
JSON Top 5:   C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\output\cybersecurity_2016_top5_costliest.json
Backup JSON:  C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\backup\cybersecurity_2016_top5_costliest_backup.json
Report TXT:   C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\output\cybersecurity_2016_incident_report.txt


In [9]:
total_records = 0
incidents_target_year = 0
attack_counts = Counter()
losses = []
selected_incidents = []

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    original_order = 0

    for row in reader:
        total_records += 1
        year_val = (row.get("Year") or "").strip()

        if year_val == str(TARGET_YEAR):
            original_order += 1
            incidents_target_year += 1

            attack_type = (row.get(COUNT_FIELD) or "Unknown").strip()
            attack_counts[attack_type] += 1

            loss_str = (row.get(LOSS_FIELD) or "").strip()
            try:
                loss_val = float(loss_str)
            except ValueError:
                loss_val = 0.0

            losses.append(loss_val)

            incident_dict = dict(row)
            incident_dict["_orig_order"] = original_order
            incident_dict["_loss_float"] = loss_val
            selected_incidents.append(incident_dict)

total_financial_loss = sum(losses)
avg_financial_loss = (total_financial_loss / len(losses)) if losses else 0.0

# Rank incidents: descending financial loss, retain original dataset order when tied
ranked_incidents = sorted(
    selected_incidents,
    key=lambda x: (-x["_loss_float"], x["_orig_order"])
)
top_5_costliest = ranked_incidents[:5]

clean_top_5 = []
for rank, inc in enumerate(top_5_costliest, 1):
    item = {k: v for k, v in inc.items() if not k.startswith("_")}
    item["Rank"] = rank
    clean_top_5.append(item)

# Export Top 5 to JSON
json_export_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "target_year": TARGET_YEAR,
        "total_records_in_dataset": total_records,
        "incidents_in_target_year": incidents_target_year,
        "count_per_attack_type": dict(attack_counts),
        "total_financial_loss_usd_millions": round(total_financial_loss, 2),
        "average_financial_loss_usd_millions": round(avg_financial_loss, 2),
        "five_costliest_incidents": clean_top_5
    }
}

with open(JSON_TOP5_PATH, mode="w", encoding="utf-8") as jf:
    json.dump(json_export_data, jf, indent=4)

# Export selected incidents to filtered CSV
csv_fieldnames = [k for k in fieldnames if not k.startswith("_")]
with open(FILTERED_CSV_PATH, mode="w", encoding="utf-8", newline="") as cf:
    writer = csv.DictWriter(cf, fieldnames=csv_fieldnames)
    writer.writeheader()
    for inc in selected_incidents:
        row_clean = {k: inc[k] for k in csv_fieldnames}
        writer.writerow(row_clean)

# Create backup of JSON
shutil.copy2(JSON_TOP5_PATH, BACKUP_JSON_PATH)
backup_verified = BACKUP_JSON_PATH.exists() and BACKUP_JSON_PATH.stat().st_size > 0
backup_size = BACKUP_JSON_PATH.stat().st_size if backup_verified else 0

# Generate Text Report
report_lines = [
    "=" * 70,
    f"CYBERSECURITY INCIDENT REPORT - FISCAL YEAR {TARGET_YEAR}",
    "=" * 70,
    f"Student: {LAST_NAME} ({STUDENT_ID})",
    f"Seed Digit: {SEED_DIGIT} | ID Sum: {ID_SUM}",
    f"Target Year: {TARGET_YEAR}",
    "-" * 70,
    "1. EXECUTIVE SUMMARY & ASSESSMENT METRICS",
    f"  Total records in dataset:             {total_records}",
    f"  Incidents occurring in {TARGET_YEAR}:            {incidents_target_year}",
    f"  Total Financial Loss:                 ${total_financial_loss:,.2f} Million USD",
    f"  Average Financial Loss per Incident:  ${avg_financial_loss:,.2f} Million USD",
    "-" * 70,
    "2. INCIDENTS BY ATTACK TYPE",
]
for att, count in sorted(attack_counts.items(), key=lambda x: -x[1]):
    pct = (count / incidents_target_year * 100) if incidents_target_year else 0
    report_lines.append(f"  - {att:<22}: {count:>3} incidents ({pct:>5.1f}%)")

report_lines.extend([
    "-" * 70,
    "3. FIVE COSTLIEST INCIDENTS OF 2016",
])
for item in clean_top_5:
    report_lines.append(
        f"  Rank {item['Rank']}: Loss = ${item['Financial Loss (in Million $)']} Million USD\n"
        f"         Attack Type:     {item['Attack Type']}\n"
        f"         Country:         {item['Country']}\n"
        f"         Target Industry: {item['Target Industry']}\n"
        f"         Affected Users:  {item['Number of Affected Users']}\n"
        f"         Defense Used:    {item['Defense Mechanism Used']}\n"
        f"         Vulnerability:   {item['Security Vulnerability Type']}\n"
    )

report_lines.extend([
    "-" * 70,
    "4. OUTPUT & ARCHIVE VERIFICATION",
    f"  Filtered CSV File:    {FILTERED_CSV_PATH.name} ({len(selected_incidents)} records)",
    f"  JSON Summary File:    {JSON_TOP5_PATH.name} ({JSON_TOP5_PATH.stat().st_size} bytes)",
    f"  Backup JSON File:     {BACKUP_JSON_PATH.name} ({backup_size} bytes)",
    f"  Backup Verification:  {'VERIFIED' if backup_verified else 'FAILED'}",
    f"  Incident Report File: {REPORT_TXT_PATH.name}",
    "=" * 70,
])

report_text = "\n".join(report_lines)
with open(REPORT_TXT_PATH, mode="w", encoding="utf-8") as rf:
    rf.write(report_text)

# Display Assessment Data Table
print("=" * 70)
print(f"{'Item':<38} | {'Result'}")
print("-" * 70)
print(f"{'Target year':<38} | {TARGET_YEAR}")
print(f"{'Total records in dataset':<38} | {total_records}")
print(f"{'Incidents in target year':<38} | {incidents_target_year}")
attack_summary_str = ", ".join(f"{k}: {v}" for k, v in sorted(attack_counts.items(), key=lambda x: -x[1]))
print(f"{'Count per attack type':<38} | {attack_summary_str}")
print(f"{'Total financial loss (USD millions)':<38} | ${total_financial_loss:,.2f} Million")
print(f"{'Average financial loss (USD millions)':<38} | ${avg_financial_loss:,.2f} Million")
print(f"{'Five costliest incidents':<38} |")
for item in clean_top_5:
    print(f"  {item['Rank']}. {item['Attack Type']} - ${item['Financial Loss (in Million $)']}M - {item['Country']} ({item['Target Industry']})")
print(f"{'Filtered CSV file':<38} | {FILTERED_CSV_PATH.name} ({len(selected_incidents)} records)")
print(f"{'JSON summary file':<38} | {JSON_TOP5_PATH.name}")
print(f"{'Backup verification':<38} | {'VERIFIED' if backup_verified else 'FAILED'} ({BACKUP_JSON_PATH.name}, {backup_size} bytes)")
print(f"{'Report file':<38} | {REPORT_TXT_PATH.name}")
print("=" * 70)
print("Final Output: Exercise 3 completed successfully.")


Item                                   | Result
----------------------------------------------------------------------
Target year                            | 2016
Total records in dataset               | 3000
Incidents in target year               | 285
Count per attack type                  | Phishing: 55, DDoS: 53, Man-in-the-Middle: 47, SQL Injection: 47, Ransomware: 42, Malware: 41
Total financial loss (USD millions)    | $13,947.26 Million
Average financial loss (USD millions)  | $48.94 Million
Five costliest incidents               |
  1. Phishing - $99.98M - Germany (Telecommunications)
  2. Ransomware - $99.88M - USA (Telecommunications)
  3. Malware - $99.29M - Brazil (IT)
  4. Ransomware - $99.19M - USA (Healthcare)
  5. Phishing - $98.72M - Australia (Education)
Filtered CSV file                      | cybersecurity_2016_filtered.csv (285 records)
JSON summary file                      | cybersecurity_2016_top5_costliest.json
Backup verification                    | VERIFI